In [1]:
!pip -q install transformers datasets accelerate scikit-learn

In [2]:
import numpy as np
import pandas as pd
import torch

from google.colab import drive

print("GPU available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

GPU available: True
GPU: Tesla T4


In [3]:
drive.mount("/content/drive")

Mounted at /content/drive


In [4]:
canonical_df = pd.read_pickle(
    "/content/drive/MyDrive/router_project_canonical.pkl"
)

splits = np.load(
    "/content/drive/MyDrive/router_split_indices.npz"
)

train_idx = splits["train_idx"]
val_idx = splits["val_idx"]
test_idx = splits["test_idx"]

print("Canonical dataset:", canonical_df.shape)

print("Train:", len(train_idx))
print("Validation:", len(val_idx))
print("Test:", len(test_idx))

Canonical dataset: (22962, 26)
Train: 18369
Validation: 2296
Test: 2297


In [5]:
MODEL_COLS = [
    "WizardLM/WizardLM-13B-V1.2",
    "claude-instant-v1",
    "claude-v1",
    "claude-v2",
    "gpt-3.5-turbo-1106",
    "gpt-4-1106-preview",
    "meta/code-llama-instruct-34b-chat",
    "meta/llama-2-70b-chat",
    "mistralai/mistral-7b-chat",
    "mistralai/mixtral-8x7b-chat",
    "zero-one-ai/Yi-34B-Chat"
]

print("Number of models:", len(MODEL_COLS))

Number of models: 11


In [6]:
X = canonical_df[
    [
        "sample_id",
        "prompt",
        "domain",
        "eval_name"
    ]
].copy()

Y = pd.DataFrame({
    model:
    canonical_df[f"{model}|performance"]
    for model in MODEL_COLS
})

C = pd.DataFrame({
    model:
    canonical_df[f"{model}|cost"]
    for model in MODEL_COLS
})

print("X:", X.shape)
print("Y:", Y.shape)
print("C:", C.shape)

X: (22962, 4)
Y: (22962, 11)
C: (22962, 11)


In [7]:
X_train = X.iloc[train_idx].reset_index(drop=True)
X_val   = X.iloc[val_idx].reset_index(drop=True)
X_test  = X.iloc[test_idx].reset_index(drop=True)

Y_train = Y.iloc[train_idx].reset_index(drop=True)
Y_val   = Y.iloc[val_idx].reset_index(drop=True)
Y_test  = Y.iloc[test_idx].reset_index(drop=True)

C_train = C.iloc[train_idx].reset_index(drop=True)
C_val   = C.iloc[val_idx].reset_index(drop=True)
C_test  = C.iloc[test_idx].reset_index(drop=True)

print("TRAIN:", X_train.shape, Y_train.shape)
print("VAL  :", X_val.shape, Y_val.shape)
print("TEST :", X_test.shape, Y_test.shape)

TRAIN: (18369, 4) (18369, 11)
VAL  : (2296, 4) (2296, 11)
TEST : (2297, 4) (2297, 11)


In [8]:
from datasets import Dataset, Features, Value, Sequence

features = Features({
    "text": Value("string"),
    "labels": Sequence(
        Value("float32"),
        length=len(MODEL_COLS)
    )
})

train_ds = Dataset.from_dict(
    {
        "text": X_train["prompt"].astype(str).tolist(),
        "labels": Y_train.astype("float32").values.tolist()
    },
    features=features
)

val_ds = Dataset.from_dict(
    {
        "text": X_val["prompt"].astype(str).tolist(),
        "labels": Y_val.astype("float32").values.tolist()
    },
    features=features
)

test_ds = Dataset.from_dict(
    {
        "text": X_test["prompt"].astype(str).tolist(),
        "labels": Y_test.astype("float32").values.tolist()
    },
    features=features
)

print(train_ds)
print(val_ds)
print(test_ds)

Dataset({
    features: ['text', 'labels'],
    num_rows: 18369
})
Dataset({
    features: ['text', 'labels'],
    num_rows: 2296
})
Dataset({
    features: ['text', 'labels'],
    num_rows: 2297
})


In [9]:
print("QUESTION:\n")
print(train_ds[0]["text"])

print("\nTARGET SCORES:\n")

for model, score in zip(
    MODEL_COLS,
    train_ds[0]["labels"]
):
    print(
        f"{model:45s} -> {score:.2f}"
    )

QUESTION:

['TASK: Solve the following grade school math problem and provide a numerical answer.\nThe following are examples of grade school math problems and answers:\nQuestion: There are 15 trees in the grove. Grove workers will plant trees in the grove today. After they are done, there will be 21 trees. How many trees did the grove workers plant today? \nAnswer: 6\n\nQuestion: If there are 3 cars in the parking lot and 2 more cars arrive, how many cars are in the parking lot? \nAnswer: 5\n\nQuestion: Leah had 32 chocolates and her sister had 42. If they ate 35, how many pieces do they have left in total? \nAnswer: 39\n\nQuestion: Jason had 20 lollipops. He gave Denny some lollipops. Now Jason has 12 lollipops. How many lollipops did Jason give to Denny?\nAnswer: 8\n\nQuestion: Shawn has five toys. For Christmas, he got two toys each from his mom and dad. How many toys does he have now? \nAnswer: 9\n\n', 'There are 100 lions in Londolozi at first. If lion cubs are born at the rate of

In [10]:
from transformers import AutoTokenizer

MODEL_NAME = "distilbert-base-uncased"

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME
)

print("Tokenizer loaded.")
print("Vocabulary size:", tokenizer.vocab_size)

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

Tokenizer loaded.
Vocabulary size: 30522


In [11]:
MAX_LENGTH = 384

In [12]:
MAX_LENGTH = 384

def tokenize_batch(batch):

    return tokenizer(
        batch["text"],
        truncation=True,
        max_length=MAX_LENGTH
    )

In [13]:
train_tokenized = train_ds.map(
    tokenize_batch,
    batched=True,
    remove_columns=["text"]
)

val_tokenized = val_ds.map(
    tokenize_batch,
    batched=True,
    remove_columns=["text"]
)

test_tokenized = test_ds.map(
    tokenize_batch,
    batched=True,
    remove_columns=["text"]
)

print(train_tokenized)
print(val_tokenized)
print(test_tokenized)

Map:   0%|          | 0/18369 [00:00<?, ? examples/s]

Map:   0%|          | 0/2296 [00:00<?, ? examples/s]

Map:   0%|          | 0/2297 [00:00<?, ? examples/s]

Dataset({
    features: ['labels', 'input_ids', 'token_type_ids', 'attention_mask'],
    num_rows: 18369
})
Dataset({
    features: ['labels', 'input_ids', 'token_type_ids', 'attention_mask'],
    num_rows: 2296
})
Dataset({
    features: ['labels', 'input_ids', 'token_type_ids', 'attention_mask'],
    num_rows: 2297
})


In [14]:
sample = train_tokenized[0]

print("Fields:")
print(sample.keys())

print("\nNumber of input tokens:")
print(len(sample["input_ids"]))

print("\nNumber of target scores:")
print(len(sample["labels"]))

print("\nLabels:")
print(sample["labels"])

Fields:
dict_keys(['labels', 'input_ids', 'token_type_ids', 'attention_mask'])

Number of input tokens:
309

Number of target scores:
11

Labels:
[0.5, 0.75, 0.75, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5]


In [15]:
for name, ds in [
    ("train", train_tokenized),
    ("validation", val_tokenized),
    ("test", test_tokenized)
]:
    if "token_type_ids" in ds.column_names:
        print("Removing token_type_ids from", name)

if "token_type_ids" in train_tokenized.column_names:
    train_tokenized = train_tokenized.remove_columns(
        ["token_type_ids"]
    )

if "token_type_ids" in val_tokenized.column_names:
    val_tokenized = val_tokenized.remove_columns(
        ["token_type_ids"]
    )

if "token_type_ids" in test_tokenized.column_names:
    test_tokenized = test_tokenized.remove_columns(
        ["token_type_ids"]
    )

print(train_tokenized.column_names)

Removing token_type_ids from train
Removing token_type_ids from validation
Removing token_type_ids from test
['labels', 'input_ids', 'attention_mask']


In [16]:
from transformers import AutoModelForSequenceClassification

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=len(MODEL_COLS),
    problem_type="regression"
)

print("Model loaded.")
print("Number of outputs:", model.config.num_labels)
print("Problem type:", model.config.problem_type)

model.safetensors: reconstructing file:   0%|          |  0.00B /  268MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
classifier.bias         | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.weight   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Model loaded.
Number of outputs: 11
Problem type: regression


In [17]:
from sklearn.metrics import (
    mean_squared_error,
    mean_absolute_error
)

def compute_metrics(eval_pred):

    predictions = eval_pred.predictions
    labels = eval_pred.label_ids

    # RouterBench scores are bounded between 0 and 1
    predictions = np.clip(
        predictions,
        0.0,
        1.0
    )

    mse = mean_squared_error(
        labels,
        predictions
    )

    mae = mean_absolute_error(
        labels,
        predictions
    )

    return {
        "mse": mse,
        "mae": mae
    }

In [18]:
from transformers import DataCollatorWithPadding

data_collator = DataCollatorWithPadding(
    tokenizer=tokenizer
)

In [20]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="/content/distilbert_router_runs",

    learning_rate=2e-5,

    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,

    num_train_epochs=3,

    weight_decay=0.01,

    # Transformers 5.x:
    # 0.1 means 10% of total training steps
    warmup_steps=0.1,

    eval_strategy="epoch",
    save_strategy="epoch",

    load_best_model_at_end=True,

    metric_for_best_model="mae",
    greater_is_better=False,

    fp16=True,

    logging_steps=100,

    save_total_limit=1,

    seed=42,

    report_to="none"
)

print("TrainingArguments created successfully.")

TrainingArguments created successfully.


In [21]:
from transformers import Trainer

trainer = Trainer(
    model=model,
    args=training_args,

    train_dataset=train_tokenized,
    eval_dataset=val_tokenized,

    data_collator=data_collator,

    compute_metrics=compute_metrics,

    processing_class=tokenizer
)

print("Trainer ready.")

Trainer ready.


In [22]:
train_result = trainer.train()

Epoch,Training Loss,Validation Loss,Mse,Mae
1,0.128258,0.131150,0.131063,0.298680
2,0.118680,0.122903,0.122872,0.264557
3,0.114646,0.122479,0.122412,0.261298


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

In [23]:
print(train_result)

TrainOutput(global_step=3447, training_loss=0.12868527062982968, metrics={'train_runtime': 527.039, 'train_samples_per_second': 104.56, 'train_steps_per_second': 6.54, 'total_flos': 5111994985888854.0, 'train_loss': 0.12868527062982968, 'epoch': 3.0})


In [24]:
validation_metrics = trainer.evaluate(
    val_tokenized
)

print("\nFINAL VALIDATION METRICS")

for key, value in validation_metrics.items():
    print(key, ":", value)

Training Loss,Validation Loss,Epoch,Mse,Mae
0.114646,0.122479,3,0.122412,0.261298



FINAL VALIDATION METRICS
eval_loss : 0.12247887253761292
eval_mse : 0.12241197377443314
eval_mae : 0.261298269033432


In [25]:
val_output = trainer.predict(
    val_tokenized
)

distilbert_val_pred = val_output.predictions

print(
    "Prediction shape:",
    distilbert_val_pred.shape
)

print(
    "Raw prediction range:",
    distilbert_val_pred.min(),
    "to",
    distilbert_val_pred.max()
)

Prediction shape: (2296, 11)
Raw prediction range: -0.08984375 to 1.0791016


In [26]:
distilbert_val_pred = np.clip(
    distilbert_val_pred,
    0.0,
    1.0
)

print(
    "Clipped prediction range:",
    distilbert_val_pred.min(),
    "to",
    distilbert_val_pred.max()
)

Clipped prediction range: 0.0 to 1.0


In [27]:
distilbert_val_mse = mean_squared_error(
    Y_val,
    distilbert_val_pred
)

distilbert_val_mae = mean_absolute_error(
    Y_val,
    distilbert_val_pred
)

print(
    "DistilBERT Validation MSE:",
    distilbert_val_mse
)

print(
    "DistilBERT Validation MAE:",
    distilbert_val_mae
)

DistilBERT Validation MSE: 0.12241197377443314
DistilBERT Validation MAE: 0.261298269033432


In [28]:
print("===== PREDICTION MODEL COMPARISON =====")

print(
    "TF-IDF Ridge MAE :",
    0.278425
)

print(
    "DistilBERT MAE   :",
    distilbert_val_mae
)

print()

if distilbert_val_mae < 0.278425:
    print("DistilBERT improves prediction MAE.")
else:
    print(
        "TF-IDF currently has lower prediction MAE."
    )

===== PREDICTION MODEL COMPARISON =====
TF-IDF Ridge MAE : 0.278425
DistilBERT MAE   : 0.261298269033432

DistilBERT improves prediction MAE.


In [29]:
SAVE_PATH = (
    "/content/drive/MyDrive/"
    "distilbert_llm_router"
)

trainer.save_model(
    SAVE_PATH
)

tokenizer.save_pretrained(
    SAVE_PATH
)

print(
    "Saved model to:",
    SAVE_PATH
)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Saved model to: /content/drive/MyDrive/distilbert_llm_router


In [30]:
SAVE_PATH = "/content/drive/MyDrive/distilbert_llm_router"

trainer.save_model(SAVE_PATH)
tokenizer.save_pretrained(SAVE_PATH)

np.save(
    "/content/drive/MyDrive/distilbert_val_predictions.npy",
    distilbert_val_pred
)

print("Model saved to:", SAVE_PATH)
print("Validation predictions saved.")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Model saved to: /content/drive/MyDrive/distilbert_llm_router
Validation predictions saved.


In [31]:
print("Final DistilBERT MAE:", distilbert_val_mae)
print("Final DistilBERT MSE:", distilbert_val_mse)

Final DistilBERT MAE: 0.261298269033432
Final DistilBERT MSE: 0.12241197377443314
